# Practica 3.1 - CNN + similitud de coseno

**Deep Learning Multimodal: CNN + Transformers**

Este notebook implementa el punto **3.1 CNN + similitud de coseno** de la practica:

1. Carga 150 posters locales de peliculas importadas desde TMDB.
2. Usa **ResNet-50 preentrenado en ImageNet** como extractor visual.
3. Genera embeddings visuales de 2048 dimensiones.
4. Calcula similitud de coseno entre peliculas.
5. Recomienda las top-K peliculas mas similares para cold start.
6. Guarda resultados en `data/processed`.

## 1. Configuracion

Si ejecutas en Colab, activa GPU. En esta maquina tambien funciona en CPU, solo tarda un poco mas.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

POSTERS_DIR = PROJECT_ROOT / "data" / "posters"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROJECT_ROOT)
print("Posters:", POSTERS_DIR)
print("Salida:", PROCESSED_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

## 2. Cargar metadatos de posters

Se usa `poster_download_log.csv` y los archivos `.jpg` de `data/posters`.

In [ ]:
import importlib.util

script_path = PROJECT_ROOT / "scripts" / "06_cnn_cosine_recommendations.py"
spec = importlib.util.spec_from_file_location("cnn_cosine", script_path)
cnn_cosine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cnn_cosine)

metadata = cnn_cosine.load_metadata()
metadata[["film_id", "title", "poster_file"]].head()

In [ ]:
print(f"Posters disponibles: {len(metadata)}")
metadata[["film_id", "title"]].head(10)

## 3. Cargar ResNet-50 preentrenado

Quitamos la capa final (`fc`) y usamos la salida anterior como embedding visual de 2048 dimensiones.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

model, transform = cnn_cosine.build_resnet50(device=device, allow_random_fallback=False)
model

## 4. Descargar/preprocesar posters

En este proyecto los posters ya estan descargados. El preprocesamiento se aplica con las transformaciones oficiales de los pesos de ResNet-50.

In [ ]:
from IPython.display import display

for poster_path in metadata["poster_file"].head(3):
    image = Image.open(poster_path).convert("RGB")
    image.thumbnail((150, 220))
    display(image)

## 5. Generar embeddings visuales

In [ ]:
dataset = cnn_cosine.PosterDataset(metadata, transform)
visual_embeddings = cnn_cosine.generate_embeddings(
    model=model,
    dataset=dataset,
    device=device,
    batch_size=4,
)

visual_embeddings.shape

In [ ]:
np.save(PROCESSED_DIR / "visual_embeddings.npy", visual_embeddings)
metadata.to_csv(PROCESSED_DIR / "movies_visual_metadata.csv", index=False)

print("Guardado:", PROCESSED_DIR / "visual_embeddings.npy")
print("Guardado:", PROCESSED_DIR / "movies_visual_metadata.csv")

## 6. Calcular matriz de similitud coseno

La similitud de coseno mide el angulo entre dos embeddings:

\[
cos(A,B)=\frac{A \cdot B}{\|A\|\|B\|}
\]

In [ ]:
visual_similarity = cosine_similarity(visual_embeddings)
np.save(PROCESSED_DIR / "visual_similarity.npy", visual_similarity)

print("Matriz:", visual_similarity.shape)
print("Similitud pelicula 0 consigo misma:", visual_similarity[0, 0])

## 7. Sistema de recomendacion cold start

Dada una pelicula, se devuelven las top-K mas similares visualmente.

In [ ]:
def recommend_visual(movie_idx, top_k=5):
    return cnn_cosine.recommend_by_index(
        movie_idx=movie_idx,
        metadata=metadata,
        similarity_matrix=visual_similarity,
        top_k=top_k,
    )

recommend_visual(0, top_k=5)[["rank", "query_title", "title", "similarity"]]

In [ ]:
all_recommendations = cnn_cosine.build_recommendation_table(
    metadata=metadata,
    similarity_matrix=visual_similarity,
    top_k=5,
    sample_size=10,
)
all_recommendations.to_csv(PROCESSED_DIR / "visual_recommendations_top5.csv", index=False)
all_recommendations.head(15)

## 8. Visualizacion cualitativa de recomendaciones

In [ ]:
import matplotlib.pyplot as plt

def show_recommendations(movie_idx, top_k=5):
    recs = recommend_visual(movie_idx, top_k=top_k)
    rows = [metadata.iloc[movie_idx]] + [
        metadata.iloc[metadata.index[metadata["film_id"] == film_id][0]]
        for film_id in recs["film_id"]
    ]
    titles = ["Consulta"] + [f"Top {rank}" for rank in recs["rank"]]

    plt.figure(figsize=(2.3 * len(rows), 4.2))
    for i, row in enumerate(rows):
        image = Image.open(row["poster_file"]).convert("RGB")
        ax = plt.subplot(1, len(rows), i + 1)
        ax.imshow(image)
        ax.set_title(f"{titles[i]}\n{row['title']}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()

show_recommendations(0, top_k=5)

## 9. t-SNE opcional

Proyecta los embeddings de 2048 dimensiones a 2D para inspeccionar grupos visuales.

In [ ]:
cnn_cosine.save_tsne_plot(
    embeddings=visual_embeddings,
    metadata=metadata,
    output_path=PROCESSED_DIR / "visual_embeddings_tsne.png",
)

img = Image.open(PROCESSED_DIR / "visual_embeddings_tsne.png")
display(img)

## Resultados generados

- `data/processed/visual_embeddings.npy`
- `data/processed/visual_similarity.npy`
- `data/processed/movies_visual_metadata.csv`
- `data/processed/visual_recommendations_top5.csv`
- `data/processed/visual_embeddings_tsne.png`

Con esto queda cubierto el punto **3.1 CNN + similitud de coseno**: modelo ResNet-50, preprocesamiento de posters, embeddings, matriz coseno, recomendador top-K y visualizacion.